# Step 01: Exploratory Data Analysis (IEEE-CIS Fraud Detection)
This notebook investigates the raw IEEE-CIS dataset, verifies class imbalance, analyzes missing values, and identifies temporal patterns to design our pipeline architecture.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT_DIR))

from src.config import RAW_DATA_DIR, TIME_COL, TARGET_COL, ID_COL
from src.data import load_raw_data, compute_eda_summary

print('Environment and modules loaded.')

## 1. Load Raw Transactions and Identity Tables with Memory Downcasting

In [ ]:
df = load_raw_data(verbose=True)
summary = compute_eda_summary(df)
for k, v in summary.items():
    print(f'{k}: {v}')

## 2. Inspect Target Variable & Class Imbalance

In [ ]:
counts = df[TARGET_COL].value_counts()
print('Counts:\n', counts)
print(f'Fraud percentage: {df[TARGET_COL].mean() * 100:.2f}%')

plt.figure(figsize=(6, 4))
counts.plot(kind='bar', color=['#2ecc71', '#e74c3c'])
plt.title('Target Distribution (0: Genuine, 1: Fraud)')
plt.ylabel('Count')
plt.show()

## 3. Analyze Missing Value Ratios

In [ ]:
missing = df.isnull().mean() * 100
print(f'Columns with >50% missing: {(missing > 50).sum()}')
print(f'Columns with >80% missing: {(missing > 80).sum()}')

plt.figure(figsize=(7, 4))
missing.plot(kind='hist', bins=25, color='#8e44ad', edgecolor='white')
plt.axvline(50, color='orange', linestyle='--', label='50% threshold')
plt.axvline(80, color='red', linestyle='--', label='80% threshold')
plt.title('Distribution of Feature Missingness (%)')
plt.xlabel('% Missing')
plt.legend()
plt.show()

## 4. Fraud Rate Over Time (Temporal Drift)

In [ ]:
df['day'] = (df[TIME_COL] / 86400).astype(int)
trend = df.groupby('day')[TARGET_COL].agg(['count', 'mean']).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()
ax1.plot(trend['day'], trend['mean'] * 100, color='#e74c3c', label='Fraud Rate %')
ax2.bar(trend['day'], trend['count'], color='#3498db', alpha=0.3, label='Volume')
ax1.set_xlabel('Day')
ax1.set_ylabel('Fraud Rate (%)', color='#e74c3c')
ax2.set_ylabel('Transaction Volume', color='#2980b9')
plt.title('Daily Transaction Volume and Fraud Rate Over 182 Days')
plt.show()